# Ch4. Time Series Features
**Forecasting: Principles & Practice (Python Edition)**  
Lab Notebook · [github.com/bcseong2/fpppy-labs](https://github.com/bcseong2/fpppy-labs)

In [ ]:
%pip install statsforecast neuralforecast hierarchicalforecast mlforecast utilsforecast

## [Slide 1] Chapter Overview

In [ ]:
import tsfeatures as tsf
from sklearn.decomposition import PCA
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

## [Slide 3] 4.1 Simple Statistics

In [ ]:
aus_tourism = pd.read_csv("data/aus_tourism.csv",
                           parse_dates=["ds"])

summary_stats = tsf.tsfeatures(
    aus_tourism,
    freq=4,
    features=[tsf.statistics],
    scale=False
)
summary_stats[["unique_id","min","p25","median","p75","max"]].head(10)

## [Slide 5] 4.2 ACF Features

In [ ]:
acf_feat = tsf.tsfeatures(
    aus_tourism,
    freq=4,
    features=[tsf.acf_features]
)
acf_feat.head(10).iloc[:, :5]

## [Slide 7] 4.3 STL Decomposition Review

In [ ]:
from statsmodels.tsa.seasonal import STL
import numpy as np

stl = STL(y, period=12, robust=True)
res = stl.fit()
# Components: res.trend, res.seasonal, res.resid

## [Slide 9] 4.3 Trend Strength F_T and Seasonal Strength F_S

In [ ]:
def compute_stl_features(y, period=12):
    res = STL(y, period=period, robust=True).fit()
    ft = max(0, 1 - np.var(res.resid) /
                   np.var(res.trend + res.resid))
    fs = max(0, 1 - np.var(res.resid) /
                   np.var(res.seasonal + res.resid))
    return {"trend_strength": ft, "seasonal_strength": fs}

## [Slide 11] 4.3 Computing STL Features with tsfeatures

In [ ]:
stl_feat = tsf.tsfeatures(
    aus_tourism,
    freq=4,
    features=[tsf.stl_features]
)
stl_feat.head(10).iloc[:, :5]

## [Slide 13] 4.3 Visualising STL Features by State

In [ ]:
df = (
    stl_feat["unique_id"].str.split("-", expand=True)
    .rename(columns={0:"region", 1:"state", 2:"purpose"})
    .join(stl_feat)
)
fig, axs = plt.subplots(3, 3, figsize=(8, 8))
axs = axs.flatten()
for ax, (state, state_df) in zip(axs, df.groupby('state')):
    sns.scatterplot(x="trend", y="seasonal_strength",
        hue="purpose", edgecolor="none",
        data=state_df, ax=ax)
    ax.set(title=state, xlim=(0,1), ylim=(0,1))

## [Slide 16] 4.4 Computing All Features

In [ ]:
all_features = [
    tsf.acf_features,
    tsf.arch_stat,
    tsf.crossing_points,
    tsf.entropy,
    tsf.flat_spots,
    tsf.heterogeneity,
    tsf.holt_parameters,
    tsf.lumpiness,
    tsf.nonlinearity,
    tsf.pacf_features,
    tsf.stl_features,
    tsf.stability,
    tsf.hw_parameters,
    tsf.unitroot_kpss,
    tsf.unitroot_pp,
    tsf.series_length,
    tsf.hurst,
]
all_feat = tsf.tsfeatures(
    aus_tourism, freq=4, features=all_features
)

## [Slide 18] 4.5 Australian Tourism Dataset

In [ ]:
aus_tourism = pd.read_csv("data/aus_tourism.csv",
                           parse_dates=["ds"])
# 304 unique series
print(aus_tourism["unique_id"].nunique())

## [Slide 20] 4.5 Pairwise Feature Plot

In [ ]:
seasonal_feat = all_feat[[
    "unique_id","seasonal_strength","peak","trough",
    "seas_acf1","seas_pacf"
]]
df = (
    seasonal_feat["unique_id"].str.split("-", expand=True)
    .rename(columns={0:"region",1:"state",2:"Purpose"})
    .join(seasonal_feat)
)
g = sns.pairplot(df, hue="Purpose")
g.fig.set_size_inches(10, 10)

## [Slide 22] 4.5 PCA on Feature Matrix

In [ ]:
df = (
    all_feat
    .assign(purpose=lambda x:
        x["unique_id"].str.split("-").str[-1])
    .dropna(axis='columns')
)
X = df.drop(columns=["unique_id","purpose"])

pipeline_pca = Pipeline([
    ('scale', StandardScaler()),
    ('pca', PCA(n_components=2)),
])
principal_components = pipeline_pca.fit_transform(X)
pca_df = (
    pd.DataFrame(principal_components,
                 columns=["PC1","PC2"])
    .join(df[["unique_id","purpose"]])
)

## [Slide 24] 4.5 PCA Results

In [ ]:
fig, ax = plt.subplots()
sns.scatterplot(x="PC1", y="PC2", s=80,
    hue="purpose", edgecolor="none",
    data=pca_df, ax=ax)
ax.set(title="PCA of Features",
       xlabel="Principal Component 1",
       ylabel="Principal Component 2")

## [Slide 26] 4.5 Feature-Based Outlier Detection

In [ ]:
lof = LocalOutlierFactor(n_neighbors=10).fit(
    pca_df[["PC1","PC2"]]
)
res = pca_df.assign(
    score=lof.negative_outlier_factor_
)
# Most negative score = most outlying
selected_ids = (
    res.sort_values("score")["unique_id"].head(3)
)

## [Slide 28] 4.5 Visualising Outlying Series

In [ ]:
fig, axs = plt.subplots(3, figsize=(8,8), sharex=True)
for ax, uid in zip(axs, selected_ids):
    df_s = aus_tourism.loc[
        lambda x: x["unique_id"] == uid
    ]
    ax.plot(df_s["ds"], df_s["y"], color="k")
    ax.set_title(uid, loc="left", size="medium")

fig.suptitle("Outlying time series in PC space")
fig.supylabel("Trips")
fig.supxlabel("Quarter")